Latest Version

In [ ]:
def letterbox(frame, target_w, target_h):
    h, w = frame.shape[:2]
    scale = min(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2

    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    return canvas

In [2]:
import cv2
import numpy as np
import pyaudio
import math
import time
from collections import deque, Counter

# ==========================================
# 1. PURE NUMPY AUDIO ENGINE (Crash-Proof)
# ==========================================
class AudioEngine:
    def __init__(self):
        self.SAMPLE_RATE = 44100
        self.p = pyaudio.PyAudio()
        
        # Audio Targets (Incoming from Vision)
        self.params = {
            "freq": 440.0, 
            "filter": 0.0, 
            "lfo_rate": 4.0, 
            "volume": 0.0, 
            "pan": 0.5
        }
        
        # Internal Smoothed Values (Current State)
        self.curr = {
            "freq": 440.0, 
            "filter": 0.0, 
            "lfo": 4.0, 
            "vol": 0.0, 
            "pan": 0.5
        }
        
        self.phase = 0.0
        self.lfo_phase = 0.0
        
        # Stereo Reverb Buffers
        self.delay_len_L = int(self.SAMPLE_RATE * 0.18) 
        self.delay_len_R = int(self.SAMPLE_RATE * 0.22)
        
        self.buff_L = np.zeros(self.delay_len_L, dtype=np.float32)
        self.buff_R = np.zeros(self.delay_len_R, dtype=np.float32)
        self.ptr_L = 0
        self.ptr_R = 0
        
        self.stream = self.p.open(
            format=pyaudio.paFloat32, channels=2, rate=self.SAMPLE_RATE, output=True, stream_callback=self.callback
        )

    def update(self, freq, filter_val, lfo_rate, vol, pan):
        self.params["freq"] = freq
        self.params["filter"] = filter_val
        self.params["lfo_rate"] = lfo_rate
        self.params["volume"] = vol
        self.params["pan"] = pan

    def callback(self, in_data, frame_count, time_info, status):
        # 1. Update & Smooth Parameters
        # Frequency snaps instantly (No smoothing)
        self.curr["freq"] = self.params["freq"] 
        
        # Smooth the rest explicitly (FIXED THE KEY ERROR HERE)
        self.curr["filter"] += (self.params["filter"] - self.curr["filter"]) * 0.1
        self.curr["lfo"]    += (self.params["lfo_rate"] - self.curr["lfo"]) * 0.1
        self.curr["vol"]    += (self.params["volume"]   - self.curr["vol"]) * 0.1
        self.curr["pan"]    += (self.params["pan"]      - self.curr["pan"]) * 0.1

        # Optimization: Return silence if volume is 0
        if self.curr["vol"] < 0.001 and np.max(np.abs(self.buff_L)) < 0.001:
            return (np.zeros(frame_count * 2, dtype=np.float32), pyaudio.paContinue)

        t = np.arange(frame_count, dtype=np.float32)

        # 2. LFO Generation
        lfo_val = np.sin(self.lfo_phase + t * 2 * np.pi * self.curr["lfo"] / self.SAMPLE_RATE) * 8.0
        self.lfo_phase += frame_count * 2 * np.pi * self.curr["lfo"] / self.SAMPLE_RATE
        
        # 3. Oscillator Generation
        mod_freq = self.curr["freq"] + lfo_val
        phase_inc = 2 * np.pi * mod_freq / self.SAMPLE_RATE
        phases = self.phase + np.cumsum(phase_inc)
        self.phase = phases[-1] % (2 * np.pi)
        
        sine = np.sin(phases)
        saw = sine + 0.5*np.sin(2*phases) + 0.25*np.sin(4*phases)
        
        # 4. Filter Mix (Crossfade)
        mono_sig = (sine * self.curr["filter"]) + (saw * (1.0 - self.curr["filter"]))
        mono_sig *= self.curr["vol"] * 0.2 

        # 5. Stereo Panning
        pan_rad = self.curr["pan"] * (np.pi / 2.0)
        sig_L = mono_sig * np.cos(pan_rad)
        sig_R = mono_sig * np.sin(pan_rad)

        # 6. Reverb (Vectorized Ring Buffers)
        # Left Channel
        indices_L = (np.arange(frame_count) + self.ptr_L) % self.delay_len_L
        delay_L = self.buff_L[indices_L]
        out_L = sig_L + (delay_L * 0.4) 
        self.buff_L[indices_L] = sig_L + (delay_L * 0.6) 
        self.ptr_L = (self.ptr_L + frame_count) % self.delay_len_L
        
        # Right Channel
        indices_R = (np.arange(frame_count) + self.ptr_R) % self.delay_len_R
        delay_R = self.buff_R[indices_R]
        out_R = sig_R + (delay_R * 0.4)
        self.buff_R[indices_R] = sig_R + (delay_R * 0.6)
        self.ptr_R = (self.ptr_R + frame_count) % self.delay_len_R
        
        # 7. Interleave
        stereo_out = np.empty(frame_count * 2, dtype=np.float32)
        stereo_out[0::2] = out_L
        stereo_out[1::2] = out_R
        
        return (stereo_out, pyaudio.paContinue)

    def close(self):
        self.stream.stop_stream(); self.stream.close(); self.p.terminate()

# ==========================================
# 2. VISION APP
# ==========================================
class VisionHarpFinal:
    def __init__(self):
        self.cap = cv2.VideoCapture(0)
        self.audio = AudioEngine()
        
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
        self.kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.statePost = np.array([0, 0, 0, 0], np.float32)
        
        self.lower_skin = None; self.upper_skin = None; self.is_calibrated = False
        self.finger_buffer = deque(maxlen=5)
        self.particles = deque(maxlen=20) 
        
        self.scale_mode = "Major"
        self.base_root = 130.81
        self.scale_data = {
            "Major": {"intervals": [0, 2, 4, 5, 7, 9, 11], "names": ["C", "D", "E", "F", "G", "A", "B"]},
            "Minor": {"intervals": [0, 2, 3, 5, 7, 8, 10], "names": ["C", "D", "Eb", "F", "G", "Ab", "Bb"]},
            "Blues": {"intervals": [0, 3, 5, 6, 7, 10, 12], "names": ["C", "Eb", "F", "F#", "G", "Bb", "C"]},
            "Hirajoshi": {"intervals": [0, 2, 3, 7, 8], "names": ["C", "D", "Eb", "G", "Ab"]}
        }
        self.current_scale = []; self.current_note_names = []
        self.update_scale()

    def update_scale(self):
        data = self.scale_data[self.scale_mode]
        pattern = data["intervals"]
        names = data["names"]
        full_scale = []
        full_names = []
        for octave in [0.5, 1.0, 2.0]:
            for i, semitone in enumerate(pattern):
                freq = (self.base_root * octave) * (2 ** (semitone / 12.0))
                full_scale.append(freq)
                oct_suffix = "2" if octave == 0.5 else ("3" if octave == 1.0 else "4")
                name_idx = i % len(names)
                full_names.append(f"{names[name_idx]}{oct_suffix}")
        self.current_scale = full_scale
        self.current_note_names = full_names

    def calibrate(self, frame, rect):
        x, y, w, h = rect
        roi = frame[y:y+h, x:x+w]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        mean = np.mean(hsv, axis=(0, 1))
        self.lower_skin = np.array([max(0, mean[0]-25), 30, 40], dtype=np.uint8)
        self.upper_skin = np.array([min(180, mean[0]+25), 255, 255], dtype=np.uint8)
        self.is_calibrated = True
        print(f"Calibrated: {mean}")

    def get_palm_center(self, mask_roi):
        dist_transform = cv2.distanceTransform(mask_roi, cv2.DIST_L2, 5)
        _, max_val, _, max_loc = cv2.minMaxLoc(dist_transform)
        return max_loc, max_val

    def robust_finger_count(self, contour, palm_center, palm_radius):
        if contour is None or len(contour) < 5: return 0
        area = cv2.contourArea(contour)
        perim = cv2.arcLength(contour, True)
        if perim == 0: return 0
        circularity = 4 * math.pi * (area / (perim ** 2))
        if circularity > 0.65: return 0 

        hull = cv2.convexHull(contour, returnPoints=False)
        if hull is None or len(hull) <= 2: return 0
        try: defects = cv2.convexityDefects(contour, hull)
        except: return 0
        count = 0
        if defects is not None:
            for i in range(defects.shape[0]):
                s, e, f, d = defects[i, 0]
                start = tuple(contour[s][0])
                depth = d / 256.0
                if depth > palm_radius * 0.3:
                    if start[1] < palm_center[1] + (palm_radius * 0.9):
                        if math.hypot(start[0]-palm_center[0], start[1]-palm_center[1]) > palm_radius * 1.3:
                            count += 1
        return min(count + 1 if count > 0 else 0, 5)

    def draw_particles(self, frame, sx, sy, color):
        self.particles.append((sx, sy))
        total_p = len(self.particles)
        for i in range(1, total_p):
            calc_thick = int((i / total_p) * 5)
            thickness = max(1, calc_thick) 
            cv2.line(frame, self.particles[i-1], self.particles[i], color, thickness)

    def draw_grid_and_notes(self, frame, H, W, active_idx):
        num_notes = len(self.current_scale)
        col_w = W / num_notes
        for i in range(num_notes):
            x_pos = int(i * col_w)
            if i == active_idx:
                line_color = (0, 255, 255); text_color = (0, 255, 255); thickness = 2
            else:
                line_color = (40, 40, 40); text_color = (100, 100, 100); thickness = 1
            cv2.line(frame, (x_pos, 0), (x_pos, H), line_color, thickness)
            y_pos = 30 if i % 2 == 0 else 50
            note_name = self.current_note_names[i]
            cv2.putText(frame, note_name, (x_pos + 5, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)

    def run(self):
        cv2.namedWindow("VisionHarp Final", cv2.WINDOW_NORMAL)
        while True:
            ret, frame = self.cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            H, W, _ = frame.shape
            
            cv2.putText(frame, f"SCALE: {self.scale_mode}", (10, H-40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, "[1]Maj [2]Min [3]Blues [4]Hirajoshi", (10, H-15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

            if not self.is_calibrated:
                cx, cy = W//2, H//2
                cv2.rectangle(frame, (cx-50, cy-50), (cx+50, cy+50), (0, 255, 0), 2)
                cv2.putText(frame, "PALM IN BOX -> PRESS 'C'", (cx-150, cy-70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('c'): self.calibrate(frame, (cx-50, cy-50, 100, 100))
                if key == ord('q'): break
            else:
                hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
                mask = cv2.inRange(hsv, self.lower_skin, self.upper_skin)
                mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3),np.uint8))
                mask = cv2.dilate(mask, np.ones((3,3),np.uint8), iterations=3)
                mask = cv2.GaussianBlur(mask, (5, 5), 0)
                
                cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                found = False; active_note_idx = -1
                
                if cnts:
                    c = max(cnts, key=cv2.contourArea)
                    if cv2.contourArea(c) > 2000:
                        found = True
                        x, y, w, h = cv2.boundingRect(c)
                        roi = mask[y:y+h, x:x+w]
                        local_palm, palm_radius = self.get_palm_center(roi)
                        gx, gy = x + local_palm[0], y + local_palm[1]
                        
                        self.kf.predict()
                        meas = np.array([[np.float32(gx)], [np.float32(gy)]])
                        self.kf.correct(meas)
                        sx, sy = int(self.kf.statePost[0]), int(self.kf.statePost[1])
                        sx, sy = max(0, min(W, sx)), max(0, min(H, sy))
                        
                        safe_radius = max(1, min(200, int(palm_radius)))
                        raw_fingers = self.robust_finger_count(c, (gx, gy), palm_radius)
                        self.finger_buffer.append(raw_fingers)
                        stable_fingers = Counter(self.finger_buffer).most_common(1)[0][0]
                        
                        norm_x = sx / W
                        active_note_idx = int(norm_x * len(self.current_scale))
                        active_note_idx = max(0, min(active_note_idx, len(self.current_scale)-1))
                        target_freq = self.current_scale[active_note_idx]
                        current_note_name = self.current_note_names[active_note_idx]
                        
                        target_filt = 1.0 - (min(stable_fingers, 5) / 5.0)
                        norm_z = np.clip((safe_radius - 10) / 50.0, 0, 1.0)
                        target_lfo = 2 ** (2.0 + (norm_z * 6.0))
                        target_pan = np.clip(norm_x, 0.0, 1.0)
                        
                        self.audio.update(target_freq, target_filt, target_lfo, 1.0, target_pan)
                        
                        color = (int(50*target_filt), 255, int(255*(1-target_filt)))
                        self.draw_particles(frame, sx, sy, color)
                        cv2.circle(frame, (sx, sy), safe_radius, color, 2)
                        cv2.putText(frame, f"{current_note_name}", (sx+15, sy), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                        
                self.draw_grid_and_notes(frame, H, W, active_note_idx)
                if not found:
                    self.audio.update(440, 0, 4, 0.0, 0.5)
                    self.particles.clear()

                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'): break
                if key == ord('r'): self.is_calibrated = False
                if key == ord('1'): self.scale_mode = "Major"; self.update_scale()
                if key == ord('2'): self.scale_mode = "Minor"; self.update_scale()
                if key == ord('3'): self.scale_mode = "Blues"; self.update_scale()
                if key == ord('4'): self.scale_mode = "Hirajoshi"; self.update_scale()

            cv2.imshow("VisionHarp Final", frame)

        self.audio.close()
        self.cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    app = VisionHarpFinal()
    app.run()

Calibrated: [ 19.9309  47.6629 183.4432]
Calibrated: [110.0312  67.8789 113.4458]
Calibrated: [110.6087  57.6047 136.8706]


Enhanced Version — FG/BG Segmentation + Quality-of-Life Improvements

In [1]:
import cv2
import numpy as np
import pyaudio
import math
import time
from collections import deque, Counter

# ==========================================
# 1. PURE NUMPY AUDIO ENGINE (Crash-Proof)
# ==========================================
class AudioEngine:
    def __init__(self):
        self.SAMPLE_RATE = 44100
        self.p = pyaudio.PyAudio()
        
        # Audio Targets (Incoming from Vision)
        self.params = {
            "freq": 440.0, 
            "filter": 0.0, 
            "lfo_rate": 4.0, 
            "volume": 0.0, 
            "pan": 0.5
        }
        
        # Internal Smoothed Values (Current State)
        self.curr = {
            "freq": 440.0, 
            "filter": 0.0, 
            "lfo": 4.0, 
            "vol": 0.0, 
            "pan": 0.5
        }
        
        self.phase = 0.0
        self.lfo_phase = 0.0
        
        # Stereo Reverb Buffers
        self.delay_len_L = int(self.SAMPLE_RATE * 0.18) 
        self.delay_len_R = int(self.SAMPLE_RATE * 0.22)
        
        self.buff_L = np.zeros(self.delay_len_L, dtype=np.float32)
        self.buff_R = np.zeros(self.delay_len_R, dtype=np.float32)
        self.ptr_L = 0
        self.ptr_R = 0
        
        self.stream = self.p.open(
            format=pyaudio.paFloat32, channels=2, rate=self.SAMPLE_RATE, output=True, stream_callback=self.callback
        )

    def update(self, freq, filter_val, lfo_rate, vol, pan):
        self.params["freq"] = freq
        self.params["filter"] = filter_val
        self.params["lfo_rate"] = lfo_rate
        self.params["volume"] = vol
        self.params["pan"] = pan

    def callback(self, in_data, frame_count, time_info, status):
        # 1. Update & Smooth Parameters
        self.curr["freq"] = self.params["freq"] 
        
        self.curr["filter"] += (self.params["filter"] - self.curr["filter"]) * 0.1
        self.curr["lfo"]    += (self.params["lfo_rate"] - self.curr["lfo"]) * 0.1
        self.curr["vol"]    += (self.params["volume"]   - self.curr["vol"]) * 0.1
        self.curr["pan"]    += (self.params["pan"]      - self.curr["pan"]) * 0.1

        # Optimization: Return silence if volume is 0
        if self.curr["vol"] < 0.001 and np.max(np.abs(self.buff_L)) < 0.001:
            return (np.zeros(frame_count * 2, dtype=np.float32), pyaudio.paContinue)

        t = np.arange(frame_count, dtype=np.float32)

        # 2. LFO Generation
        lfo_val = np.sin(self.lfo_phase + t * 2 * np.pi * self.curr["lfo"] / self.SAMPLE_RATE) * 8.0
        self.lfo_phase += frame_count * 2 * np.pi * self.curr["lfo"] / self.SAMPLE_RATE
        
        # 3. Oscillator Generation
        mod_freq = self.curr["freq"] + lfo_val
        phase_inc = 2 * np.pi * mod_freq / self.SAMPLE_RATE
        phases = self.phase + np.cumsum(phase_inc)
        self.phase = phases[-1] % (2 * np.pi)
        
        sine = np.sin(phases)
        saw = sine + 0.5*np.sin(2*phases) + 0.25*np.sin(4*phases)
        
        # 4. Filter Mix (Crossfade)
        mono_sig = (sine * self.curr["filter"]) + (saw * (1.0 - self.curr["filter"]))
        mono_sig *= self.curr["vol"] * 0.2 

        # 5. Stereo Panning
        pan_rad = self.curr["pan"] * (np.pi / 2.0)
        sig_L = mono_sig * np.cos(pan_rad)
        sig_R = mono_sig * np.sin(pan_rad)

        # 6. Reverb (Vectorized Ring Buffers)
        indices_L = (np.arange(frame_count) + self.ptr_L) % self.delay_len_L
        delay_L = self.buff_L[indices_L]
        out_L = sig_L + (delay_L * 0.4) 
        self.buff_L[indices_L] = sig_L + (delay_L * 0.6) 
        self.ptr_L = (self.ptr_L + frame_count) % self.delay_len_L
        
        indices_R = (np.arange(frame_count) + self.ptr_R) % self.delay_len_R
        delay_R = self.buff_R[indices_R]
        out_R = sig_R + (delay_R * 0.4)
        self.buff_R[indices_R] = sig_R + (delay_R * 0.6)
        self.ptr_R = (self.ptr_R + frame_count) % self.delay_len_R
        
        # 7. Interleave
        stereo_out = np.empty(frame_count * 2, dtype=np.float32)
        stereo_out[0::2] = out_L
        stereo_out[1::2] = out_R
        
        return (stereo_out, pyaudio.paContinue)

    def close(self):
        self.stream.stop_stream(); self.stream.close(); self.p.terminate()

# ==========================================
# 2. VISION APP — ENHANCED
# ==========================================
# New features:
#   - Foreground/Background segmentation via background-blur
#   - Adaptive skin detection with running average background model
#   - Velocity-based hand gesture dynamics (faster hand -> louder/brighter)
#   - FPS counter overlay
#   - Visual waveform meter in the HUD
#   - Improved morphological pipeline for cleaner skin mask
# ==========================================
class VisionHarpEnhanced:
    def __init__(self):
        self.cap = cv2.VideoCapture(0)
        self.audio = AudioEngine()
        
        # Kalman Filter for smooth tracking
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
        self.kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.statePost = np.array([0, 0, 0, 0], np.float32)
        
        self.lower_skin = None; self.upper_skin = None; self.is_calibrated = False
        self.finger_buffer = deque(maxlen=7)  # Increased buffer for more stable count
        self.particles = deque(maxlen=30)     # Longer trail
        
        # ---- NEW: Background model for FG/BG segmentation ----
        self.bg_model = None          # Running average background (float32)
        self.bg_alpha = 0.005         # Learning rate for background update
        self.blur_strength = 35       # Background blur kernel size (must be odd)
        self.seg_enabled = True       # Toggle with 'b' key
        
        # ---- NEW: Velocity tracking ----
        self.prev_pos = None
        self.velocity = 0.0
        
        # ---- NEW: FPS counter ----
        self.fps_timer = time.time()
        self.fps_count = 0
        self.fps_display = 0.0
        
        # Musical scales
        self.scale_mode = "Major"
        self.base_root = 130.81
        self.scale_data = {
            "Major": {"intervals": [0, 2, 4, 5, 7, 9, 11], "names": ["C", "D", "E", "F", "G", "A", "B"]},
            "Minor": {"intervals": [0, 2, 3, 5, 7, 8, 10], "names": ["C", "D", "Eb", "F", "G", "Ab", "Bb"]},
            "Blues": {"intervals": [0, 3, 5, 6, 7, 10, 12], "names": ["C", "Eb", "F", "F#", "G", "Bb", "C"]},
            "Pentatonic": {"intervals": [0, 2, 4, 7, 9], "names": ["C", "D", "E", "G", "A"]},
            "Hirajoshi": {"intervals": [0, 2, 3, 7, 8], "names": ["C", "D", "Eb", "G", "Ab"]}
        }
        self.current_scale = []; self.current_note_names = []
        self.update_scale()

    def update_scale(self):
        data = self.scale_data[self.scale_mode]
        pattern = data["intervals"]
        names = data["names"]
        full_scale = []
        full_names = []
        for octave in [0.5, 1.0, 2.0]:
            for i, semitone in enumerate(pattern):
                freq = (self.base_root * octave) * (2 ** (semitone / 12.0))
                full_scale.append(freq)
                oct_suffix = "2" if octave == 0.5 else ("3" if octave == 1.0 else "4")
                name_idx = i % len(names)
                full_names.append(f"{names[name_idx]}{oct_suffix}")
        self.current_scale = full_scale
        self.current_note_names = full_names

    def calibrate(self, frame, rect):
        x, y, w, h = rect
        roi = frame[y:y+h, x:x+w]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        mean = np.mean(hsv, axis=(0, 1))
        std  = np.std(hsv, axis=(0, 1))
        # Adaptive thresholds using mean ± k*std (more robust than fixed offsets)
        k = 1.8
        self.lower_skin = np.array([max(0, mean[0] - k*std[0] - 10), 30, 40], dtype=np.uint8)
        self.upper_skin = np.array([min(180, mean[0] + k*std[0] + 10), 255, 255], dtype=np.uint8)
        self.is_calibrated = True
        # Initialise background model from current frame
        self.bg_model = frame.astype(np.float32)
        print(f"Calibrated — HSV mean: {mean.astype(int)}, std: {std.astype(int)}")

    # ---- NEW: Foreground/Background segmentation with blur ----
    def segment_foreground(self, frame, mask):
        """
        Blurs the background while keeping the detected foreground (hand) sharp.
        Uses the skin mask expanded slightly as the foreground region.
        Also maintains a running-average background model to improve segmentation.
        """
        # Update running average background model (only non-skin pixels)
        if self.bg_model is not None:
            inv_mask = cv2.bitwise_not(mask)
            inv_mask_3ch = cv2.merge([inv_mask, inv_mask, inv_mask])
            # Blend background regions into model
            bg_float = frame.astype(np.float32)
            self.bg_model = np.where(
                inv_mask_3ch > 0,
                self.bg_model * (1 - self.bg_alpha) + bg_float * self.bg_alpha,
                self.bg_model
            )

        # Create a smoother foreground mask for blending
        fg_mask = cv2.dilate(mask, np.ones((15, 15), np.uint8), iterations=2)
        fg_mask = cv2.GaussianBlur(fg_mask, (21, 21), 0)
        fg_mask_norm = (fg_mask.astype(np.float32) / 255.0)
        fg_mask_3ch = cv2.merge([fg_mask_norm, fg_mask_norm, fg_mask_norm])

        # Heavy Gaussian blur for background
        ks = self.blur_strength if self.blur_strength % 2 == 1 else self.blur_strength + 1
        blurred = cv2.GaussianBlur(frame, (ks, ks), 0)

        # Composite: sharp foreground + blurred background
        composite = (frame.astype(np.float32) * fg_mask_3ch +
                     blurred.astype(np.float32) * (1 - fg_mask_3ch))
        return composite.astype(np.uint8)

    def get_palm_center(self, mask_roi):
        dist_transform = cv2.distanceTransform(mask_roi, cv2.DIST_L2, 5)
        _, max_val, _, max_loc = cv2.minMaxLoc(dist_transform)
        return max_loc, max_val

    def robust_finger_count(self, contour, palm_center, palm_radius):
        if contour is None or len(contour) < 5: return 0
        area = cv2.contourArea(contour)
        perim = cv2.arcLength(contour, True)
        if perim == 0: return 0
        circularity = 4 * math.pi * (area / (perim ** 2))
        if circularity > 0.65: return 0 

        hull = cv2.convexHull(contour, returnPoints=False)
        if hull is None or len(hull) <= 2: return 0
        try: defects = cv2.convexityDefects(contour, hull)
        except: return 0
        count = 0
        if defects is not None:
            for i in range(defects.shape[0]):
                s, e, f, d = defects[i, 0]
                start = tuple(contour[s][0])
                depth = d / 256.0
                if depth > palm_radius * 0.3:
                    if start[1] < palm_center[1] + (palm_radius * 0.9):
                        if math.hypot(start[0]-palm_center[0], start[1]-palm_center[1]) > palm_radius * 1.3:
                            count += 1
        return min(count + 1 if count > 0 else 0, 5)

    def draw_particles(self, frame, sx, sy, color):
        self.particles.append((sx, sy))
        total_p = len(self.particles)
        for i in range(1, total_p):
            alpha = i / total_p
            calc_thick = max(1, int(alpha * 6))
            # Fade color along the trail
            faded = tuple(int(c * alpha) for c in color)
            cv2.line(frame, self.particles[i-1], self.particles[i], faded, calc_thick)

    def draw_grid_and_notes(self, frame, H, W, active_idx):
        num_notes = len(self.current_scale)
        col_w = W / num_notes
        overlay = frame.copy()
        for i in range(num_notes):
            x_pos = int(i * col_w)
            if i == active_idx:
                # Highlight the active column with a translucent rectangle
                cv2.rectangle(overlay, (x_pos, 0), (int((i+1)*col_w), H), (0, 255, 255), -1)
                line_color = (0, 255, 255); text_color = (0, 255, 255); thickness = 2
            else:
                line_color = (50, 50, 50); text_color = (120, 120, 120); thickness = 1
            cv2.line(overlay, (x_pos, 0), (x_pos, H), line_color, thickness)
            y_pos = 30 if i % 2 == 0 else 50
            note_name = self.current_note_names[i]
            cv2.putText(overlay, note_name, (x_pos + 5, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)
        # Blend the highlight overlay at low opacity
        cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, frame)
        # Re-draw grid lines on top so they stay crisp
        for i in range(num_notes):
            x_pos = int(i * col_w)
            lc = (0, 255, 255) if i == active_idx else (50, 50, 50)
            th = 2 if i == active_idx else 1
            cv2.line(frame, (x_pos, 0), (x_pos, H), lc, th)
            y_pos = 30 if i % 2 == 0 else 50
            tc = (0, 255, 255) if i == active_idx else (120, 120, 120)
            cv2.putText(frame, self.current_note_names[i], (x_pos + 5, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, tc, 1)

    # ---- NEW: Draw a small waveform meter ----
    def draw_waveform_meter(self, frame, H, W, active_freq, volume):
        """Draws a small animated waveform indicator in the top-right corner."""
        meter_w, meter_h = 120, 40
        ox, oy = W - meter_w - 15, 10
        # Semi-transparent background
        sub = frame[oy:oy+meter_h, ox:ox+meter_w]
        if sub.shape[0] == meter_h and sub.shape[1] == meter_w:
            dark = (sub * 0.3).astype(np.uint8)
            frame[oy:oy+meter_h, ox:ox+meter_w] = dark
            cv2.rectangle(frame, (ox, oy), (ox+meter_w, oy+meter_h), (80, 80, 80), 1)
        # Draw sine preview
        if volume > 0.01:
            t_norm = np.linspace(0, 4 * np.pi, meter_w)
            # Use velocity to modulate amplitude visually
            amp = int(meter_h * 0.35 * min(volume, 1.0))
            wave_y = (np.sin(t_norm + time.time() * 5) * amp).astype(int)
            cy = oy + meter_h // 2
            pts = [(ox + i, cy + wave_y[i]) for i in range(meter_w)]
            for i in range(1, len(pts)):
                cv2.line(frame, pts[i-1], pts[i], (0, 255, 180), 1)

    # ---- NEW: Draw HUD with FPS, velocity, segmentation status ----
    def draw_hud(self, frame, H, W):
        # FPS
        self.fps_count += 1
        elapsed = time.time() - self.fps_timer
        if elapsed >= 1.0:
            self.fps_display = self.fps_count / elapsed
            self.fps_count = 0
            self.fps_timer = time.time()
        cv2.putText(frame, f"FPS: {self.fps_display:.1f}", (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        # Segmentation status
        seg_text = "BG-Blur: ON" if self.seg_enabled else "BG-Blur: OFF"
        seg_color = (0, 255, 180) if self.seg_enabled else (100, 100, 100)
        cv2.putText(frame, seg_text, (10, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.45, seg_color, 1)
        # Velocity bar
        bar_x, bar_y, bar_w, bar_h = W - 30, 60, 15, 150
        cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (60, 60, 60), 1)
        vel_fill = int(np.clip(self.velocity / 25.0, 0, 1.0) * bar_h)
        if vel_fill > 0:
            cv2.rectangle(frame, (bar_x+1, bar_y + bar_h - vel_fill), (bar_x + bar_w - 1, bar_y + bar_h), (0, 200, 255), -1)
        cv2.putText(frame, "VEL", (bar_x - 2, bar_y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 200, 255), 1)

    def run(self):
        cv2.namedWindow("VisionHarp Enhanced", cv2.WINDOW_NORMAL)
        while True:
            ret, frame = self.cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            H, W, _ = frame.shape
            
            # Help text at bottom
            cv2.putText(frame, f"SCALE: {self.scale_mode}", (10, H-55), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, "[1]Maj [2]Min [3]Blues [4]Penta [5]Hira", (10, H-30), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 150), 1)
            cv2.putText(frame, "[B]Toggle BG-Blur  [R]Recalibrate  [Q]Quit", (10, H-10), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (130, 130, 130), 1)

            if not self.is_calibrated:
                cx, cy = W//2, H//2
                cv2.rectangle(frame, (cx-50, cy-50), (cx+50, cy+50), (0, 255, 0), 2)
                cv2.putText(frame, "PALM IN BOX -> PRESS 'C'", (cx-150, cy-70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('c'): self.calibrate(frame, (cx-50, cy-50, 100, 100))
                if key == ord('q'): break
            else:
                hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
                mask = cv2.inRange(hsv, self.lower_skin, self.upper_skin)
                # Improved morphological pipeline
                kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
                kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
                mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open, iterations=1)
                mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close, iterations=2)
                mask = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=2)
                mask = cv2.GaussianBlur(mask, (7, 7), 0)
                # Re-threshold after blur to get clean binary mask
                _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

                # ---- Foreground/Background segmentation ----
                if self.seg_enabled:
                    frame = self.segment_foreground(frame, mask)
                
                cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                found = False; active_note_idx = -1
                
                if cnts:
                    c = max(cnts, key=cv2.contourArea)
                    if cv2.contourArea(c) > 2000:
                        found = True
                        x, y, w, h = cv2.boundingRect(c)
                        roi = mask[y:y+h, x:x+w]
                        local_palm, palm_radius = self.get_palm_center(roi)
                        gx, gy = x + local_palm[0], y + local_palm[1]
                        
                        self.kf.predict()
                        meas = np.array([[np.float32(gx)], [np.float32(gy)]])
                        self.kf.correct(meas)
                        sx, sy = int(self.kf.statePost[0]), int(self.kf.statePost[1])
                        sx, sy = max(0, min(W, sx)), max(0, min(H, sy))
                        
                        # ---- Velocity calculation ----
                        if self.prev_pos is not None:
                            dx = sx - self.prev_pos[0]
                            dy = sy - self.prev_pos[1]
                            self.velocity = self.velocity * 0.7 + math.hypot(dx, dy) * 0.3
                        self.prev_pos = (sx, sy)
                        
                        safe_radius = max(1, min(200, int(palm_radius)))
                        raw_fingers = self.robust_finger_count(c, (gx, gy), palm_radius)
                        self.finger_buffer.append(raw_fingers)
                        stable_fingers = Counter(self.finger_buffer).most_common(1)[0][0]
                        
                        norm_x = sx / W
                        active_note_idx = int(norm_x * len(self.current_scale))
                        active_note_idx = max(0, min(active_note_idx, len(self.current_scale)-1))
                        target_freq = self.current_scale[active_note_idx]
                        current_note_name = self.current_note_names[active_note_idx]
                        
                        target_filt = 1.0 - (min(stable_fingers, 5) / 5.0)
                        norm_z = np.clip((safe_radius - 10) / 50.0, 0, 1.0)
                        target_lfo = 2 ** (2.0 + (norm_z * 6.0))
                        target_pan = np.clip(norm_x, 0.0, 1.0)
                        
                        # Velocity influences volume slightly (louder with fast gestures)
                        vel_boost = np.clip(self.velocity / 30.0, 0.0, 0.3)
                        target_vol = min(1.0, 0.7 + vel_boost)
                        
                        self.audio.update(target_freq, target_filt, target_lfo, target_vol, target_pan)
                        
                        color = (int(50*target_filt), 255, int(255*(1-target_filt)))
                        self.draw_particles(frame, sx, sy, color)
                        cv2.circle(frame, (sx, sy), safe_radius, color, 2)
                        # Glow ring based on velocity
                        if self.velocity > 5:
                            glow_r = safe_radius + int(self.velocity * 0.5)
                            cv2.circle(frame, (sx, sy), glow_r, color, 1)
                        cv2.putText(frame, f"{current_note_name}", (sx+15, sy-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
                        cv2.putText(frame, f"Fingers: {stable_fingers}", (sx+15, sy+20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
                        
                self.draw_grid_and_notes(frame, H, W, active_note_idx)
                self.draw_waveform_meter(frame, H, W, 
                                         self.current_scale[active_note_idx] if active_note_idx >= 0 else 0,
                                         self.audio.curr["vol"])
                if not found:
                    self.audio.update(440, 0, 4, 0.0, 0.5)
                    self.particles.clear()
                    self.prev_pos = None
                    self.velocity *= 0.8  # Decay velocity when hand is gone

                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'): break
                if key == ord('r'): self.is_calibrated = False; self.bg_model = None
                if key == ord('b'): self.seg_enabled = not self.seg_enabled
                if key == ord('1'): self.scale_mode = "Major"; self.update_scale()
                if key == ord('2'): self.scale_mode = "Minor"; self.update_scale()
                if key == ord('3'): self.scale_mode = "Blues"; self.update_scale()
                if key == ord('4'): self.scale_mode = "Pentatonic"; self.update_scale()
                if key == ord('5'): self.scale_mode = "Hirajoshi"; self.update_scale()

            # Draw HUD on every frame
            self.draw_hud(frame, H, W)
            cv2.imshow("VisionHarp Enhanced", frame)

        self.audio.close()
        self.cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    app = VisionHarpEnhanced()
    app.run()


Calibrated — HSV mean: [ 10  45 233], std: [13 52  8]
Calibrated — HSV mean: [ 12  51 233], std: [12 51 10]
Calibrated — HSV mean: [  8  45 228], std: [12 52 10]


In [8]:
import cv2
import numpy as np
import pyaudio
import math
import time
from collections import deque

# ==========================================
# 1. AUDIO ENGINE
# ==========================================
class AudioEngine:
    def __init__(self):
        self.SAMPLE_RATE = 44100
        self.p = pyaudio.PyAudio()
        self.params = {"freq": 440.0, "filter": 0.0, "lfo_rate": 4.0, "volume": 0.0, "pan": 0.5}
        self.curr   = {"freq": 440.0, "filter": 0.0, "lfo": 4.0, "vol": 0.0, "pan": 0.5}
        self.phase = 0.0; self.lfo_phase = 0.0
        self.delay_len_L = int(self.SAMPLE_RATE * 0.18)
        self.delay_len_R = int(self.SAMPLE_RATE * 0.22)
        self.buff_L = np.zeros(self.delay_len_L, dtype=np.float32)
        self.buff_R = np.zeros(self.delay_len_R, dtype=np.float32)
        self.ptr_L = 0; self.ptr_R = 0
        self.stream = self.p.open(format=pyaudio.paFloat32, channels=2,
                                  rate=self.SAMPLE_RATE, output=True,
                                  stream_callback=self.callback)

    def update(self, freq, filt, lfo, vol, pan):
        self.params["freq"] = freq
        self.params["filter"] = filt
        self.params["lfo_rate"] = lfo
        self.params["volume"] = vol
        self.params["pan"] = pan

    def callback(self, in_data, frame_count, time_info, status):
        self.curr["freq"] = self.params["freq"]
        self.curr["filter"] += (self.params["filter"] - self.curr["filter"]) * 0.1
        self.curr["lfo"]    += (self.params["lfo_rate"] - self.curr["lfo"]) * 0.1
        self.curr["vol"]    += (self.params["volume"]   - self.curr["vol"]) * 0.1
        self.curr["pan"]    += (self.params["pan"]      - self.curr["pan"]) * 0.1
        if self.curr["vol"] < 0.001 and np.max(np.abs(self.buff_L)) < 0.001:
            return (np.zeros(frame_count * 2, dtype=np.float32), pyaudio.paContinue)
        t = np.arange(frame_count, dtype=np.float32)
        lfo_v = np.sin(self.lfo_phase + t * 2*np.pi*self.curr["lfo"]/self.SAMPLE_RATE) * 8.0
        self.lfo_phase += frame_count * 2*np.pi*self.curr["lfo"]/self.SAMPLE_RATE
        mod_f = self.curr["freq"] + lfo_v
        pi = 2*np.pi*mod_f/self.SAMPLE_RATE
        phases = self.phase + np.cumsum(pi)
        self.phase = phases[-1] % (2*np.pi)
        sine = np.sin(phases)
        saw = sine + 0.5*np.sin(2*phases) + 0.25*np.sin(4*phases)
        mono = (sine*self.curr["filter"] + saw*(1-self.curr["filter"])) * self.curr["vol"] * 0.2
        pr = self.curr["pan"] * (np.pi/2)
        sL = mono*np.cos(pr); sR = mono*np.sin(pr)
        iL = (np.arange(frame_count)+self.ptr_L)%self.delay_len_L
        dL = self.buff_L[iL]; oL = sL+dL*0.4; self.buff_L[iL] = sL+dL*0.6
        self.ptr_L = (self.ptr_L+frame_count)%self.delay_len_L
        iR = (np.arange(frame_count)+self.ptr_R)%self.delay_len_R
        dR = self.buff_R[iR]; oR = sR+dR*0.4; self.buff_R[iR] = sR+dR*0.6
        self.ptr_R = (self.ptr_R+frame_count)%self.delay_len_R
        out = np.empty(frame_count*2, dtype=np.float32)
        out[0::2] = oL; out[1::2] = oR
        return (out, pyaudio.paContinue)

    def close(self):
        self.stream.stop_stream(); self.stream.close(); self.p.terminate()


# =====================================================
# 2. VISION APP — V4 (Stable Detection + Octave Control)
# =====================================================
# Changes from V3:
#   - REMOVED MOG2 (main jitter source — mask flickers as it relearns)
#   - Back to stable color-only skin detection (HSV + YCrCb fusion)
#   - Stronger Kalman smoothing (lower process noise)
#   - Position EMA smoothing on top of Kalman
#   - Note index hysteresis (must move >30% of column width to switch)
#   - Blur runs on a CACHED background, updated only every 8 frames
#   - Detection runs on the ORIGINAL frame, not the blurred composite
#   - Octave control: Up/Down arrows, default octave 3
# =====================================================
class VisionHarpV4:

    PROC_W, PROC_H = 320, 240

    def __init__(self):
        self.cap = cv2.VideoCapture(0)
        # Native resolution
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 9999)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 9999)
        self.native_w = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.native_h = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.native_w)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.native_h)
        self.sx_ratio = self.native_w / self.PROC_W
        self.sy_ratio = self.native_h / self.PROC_H

        self.audio = AudioEngine()

        # Kalman — lower process noise = smoother tracking
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
        self.kf.transitionMatrix  = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
        self.kf.processNoiseCov   = np.eye(4, dtype=np.float32) * 0.01  # was 0.03
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 5.0  # trust prediction more
        self.kf.statePost         = np.array([0,0,0,0], np.float32)

        # Skin calibration
        self.lower_skin = None; self.upper_skin = None
        self.ycrcb_lo = None; self.ycrcb_hi = None
        self.is_calibrated = False

        # Depth range (contour area as proxy for distance)
        self.ref_area = None         # calibrated hand area on small frame
        self.depth_min = 0.4         # accept 40% of ref area (hand far away)
        self.depth_max = 2.5         # accept 250% of ref area (hand close)
        self.depth_status = ""       # "OK", "TOO FAR", "TOO CLOSE"

        # Pre-allocated kernels
        self.kern_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        self.kern_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        self.kern_dilate_small = np.ones((3, 3), np.uint8)
        self.kern_dilate_fg = np.ones((9, 9), np.uint8)

        self.particles     = deque(maxlen=25)

        # Smoothed position (EMA on top of Kalman)
        self.smooth_x = 0.0
        self.smooth_y = 0.0
        self.ema_alpha = 0.35  # lower = smoother, higher = more responsive

        # Note hysteresis
        self.last_note_idx = -1

        # BG blur segmentation
        self.seg_enabled = True
        self.blurred_bg = None
        self.bg_counter = 0

        # FPS
        self.fps_t = time.time(); self.fps_n = 0; self.fps_v = 0.0

        # ---- Octave control ----
        self.octave = 3          # default octave (C3 = 130.81 Hz)
        self.min_octave = 1
        self.max_octave = 6

        # Scales
        self.scale_mode = "Major"
        self.scale_data = {
            "Major":     {"i": [0,2,4,5,7,9,11],  "n": ["C","D","E","F","G","A","B"]},
            "Minor":     {"i": [0,2,3,5,7,8,10],  "n": ["C","D","Eb","F","G","Ab","Bb"]},
            "Blues":     {"i": [0,3,5,6,7,10,12], "n": ["C","Eb","F","F#","G","Bb","C"]},
            "Pentatonic":{"i": [0,2,4,7,9],       "n": ["C","D","E","G","A"]},
            "Hirajoshi": {"i": [0,2,3,7,8],       "n": ["C","D","Eb","G","Ab"]},
        }
        self.scale = []; self.note_names = []
        self._build_scale()
        self.active_note = ""

    def _base_freq(self):
        """Base frequency for current octave. C3=130.81, C4=261.63, etc."""
        return 130.81 * (2 ** (self.octave - 3))

    def _build_scale(self):
        d = self.scale_data[self.scale_mode]
        base = self._base_freq()
        s, n = [], []
        for oct_offset, suf in [(-1, str(self.octave-1)), (0, str(self.octave)), (1, str(self.octave+1))]:
            freq_mult = 2.0 ** oct_offset
            for i, semi in enumerate(d["i"]):
                s.append((base * freq_mult) * (2 ** (semi / 12.0)))
                n.append(f"{d['n'][i % len(d['n'])]}{suf}")
        self.scale = s; self.note_names = n
        self.last_note_idx = -1  # reset hysteresis on scale change

    # ---------- calibration (simple + robust, from original) ----------
    def calibrate(self, frame, rect):
        x, y, w, h = rect
        roi = frame[y:y+h, x:x+w]
        # HSV range
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        mean = np.mean(hsv, axis=(0, 1))
        self.lower_skin = np.array([max(0, mean[0]-25), 30, 40], dtype=np.uint8)
        self.upper_skin = np.array([min(180, mean[0]+25), 255, 255], dtype=np.uint8)
        # YCrCb range (extra robustness)
        ycr = cv2.cvtColor(roi, cv2.COLOR_BGR2YCrCb)
        cr_m, cr_s = np.mean(ycr[:,:,1]), np.std(ycr[:,:,1])
        cb_m, cb_s = np.mean(ycr[:,:,2]), np.std(ycr[:,:,2])
        k = 2.0
        self.ycrcb_lo = np.array([0, max(0,int(cr_m-k*cr_s)), max(0,int(cb_m-k*cb_s))], np.uint8)
        self.ycrcb_hi = np.array([255, min(255,int(cr_m+k*cr_s)), min(255,int(cb_m+k*cb_s))], np.uint8)
        self.is_calibrated = True
        # Init blurred bg
        self.blurred_bg = cv2.blur(frame, (25, 25))
        # Reset smoothed position
        self.smooth_x = self.native_w / 2.0
        self.smooth_y = self.native_h / 2.0
        # Calibrate depth reference: measure hand area on small frame
        small_cal = cv2.resize(frame, (self.PROC_W, self.PROC_H), interpolation=cv2.INTER_LINEAR)
        cal_mask = self.build_mask_small(small_cal)
        cal_cnts, _ = cv2.findContours(cal_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cal_cnts:
            biggest = max(cal_cnts, key=cv2.contourArea)
            self.ref_area = cv2.contourArea(biggest)
        else:
            self.ref_area = 5000  # fallback default
        print(f"Calibrated — HSV mean: {mean.astype(int)}  Octave: {self.octave}  "
              f"Ref area: {self.ref_area:.0f}  Res: {self.native_w}x{self.native_h}")

    # ---------- skin mask (stable, NO MOG2) ----------
    def build_mask_small(self, small):
        """Color-only detection on 320x240. No MOG2 = no flicker."""
        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, self.lower_skin, self.upper_skin)
        # Fuse with YCrCb for extra robustness
        if self.ycrcb_lo is not None:
            ycr = cv2.cvtColor(small, cv2.COLOR_BGR2YCrCb)
            ycr_mask = cv2.inRange(ycr, self.ycrcb_lo, self.ycrcb_hi)
            mask = cv2.bitwise_and(mask, ycr_mask)
        # Clean morphology (from original code, proven stable)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, self.kern_open)
        mask = cv2.dilate(mask, self.kern_dilate_small, iterations=3)
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        return mask

    # ---------- FG/BG segmentation (fast binary, cached blur) ----------
    def segment_fg(self, frame, mask_small):
        """Binary mask composite. Blur is cached and updated rarely."""
        mask_full = cv2.resize(mask_small, (self.native_w, self.native_h),
                               interpolation=cv2.INTER_NEAREST)
        mask_full = cv2.dilate(mask_full, self.kern_dilate_fg, iterations=2)
        _, mask_full = cv2.threshold(mask_full, 127, 255, cv2.THRESH_BINARY)
        # Update cached blur every 8 frames
        self.bg_counter += 1
        if self.blurred_bg is None or self.blurred_bg.shape != frame.shape:
            self.blurred_bg = cv2.blur(frame, (25, 25))
        elif self.bg_counter % 8 == 0:
            self.blurred_bg = cv2.blur(frame, (25, 25))
        result = self.blurred_bg.copy()
        cv2.copyTo(frame, mask_full, result)
        return result

    # ---------- palm / finger (from original, proven) ----------
    def get_palm_center(self, mask_roi):
        dt = cv2.distanceTransform(mask_roi, cv2.DIST_L2, 5)
        _, mv, _, ml = cv2.minMaxLoc(dt)
        return ml, mv



    # ---------- note index with hysteresis ----------
    def get_note_index(self, norm_x, num_notes):
        """Only switch note if position crosses >30% into the new column."""
        raw_idx = int(norm_x * num_notes)
        raw_idx = max(0, min(raw_idx, num_notes - 1))
        if self.last_note_idx < 0:
            self.last_note_idx = raw_idx
            return raw_idx
        # Position within the current column (0..1)
        col_w = 1.0 / num_notes
        col_center = (self.last_note_idx + 0.5) * col_w
        dist = abs(norm_x - col_center)
        # Only switch if we've moved past 60% of column width from center
        if dist > col_w * 0.6:
            self.last_note_idx = raw_idx
        return self.last_note_idx

    # ---------- drawing ----------
    def draw_particles(self, frame, sx, sy, color):
        self.particles.append((sx, sy))
        n = len(self.particles)
        for i in range(1, n):
            t = max(1, int((i/n)*4))
            cv2.line(frame, self.particles[i-1], self.particles[i], color, t)

    def draw_ui(self, frame, H, W, active_idx):
        num = len(self.scale)
        cw = W / num

        # Top bar
        bar_h = 60
        cv2.rectangle(frame, (0, 0), (W, bar_h), (20, 20, 20), -1)
        for i in range(num):
            xp = int(i * cw)
            is_a = (i == active_idx)
            if is_a:
                cv2.rectangle(frame, (xp, 0), (int((i+1)*cw), bar_h), (40, 40, 40), -1)
                cv2.line(frame, (xp, bar_h), (int((i+1)*cw), bar_h), (0, 255, 255), 3)
            if i % 2 == 0 or is_a:
                gc = (0, 200, 200) if is_a else (35, 35, 35)
                cv2.line(frame, (xp, bar_h), (xp, H-55), gc, 1)
            tc = (0, 255, 255) if is_a else (130, 130, 130)
            fs = 0.7 if is_a else 0.55
            th = 2 if is_a else 1
            cv2.putText(frame, self.note_names[i], (xp+4, 40 if i%2==0 else 22),
                        cv2.FONT_HERSHEY_SIMPLEX, fs, tc, th, cv2.LINE_AA)

        # Bottom bar
        bot_h = 55
        cv2.rectangle(frame, (0, H-bot_h), (W, H), (20, 20, 20), -1)
        # Scale + Octave
        cv2.putText(frame, f"{self.scale_mode.upper()}  OCT:{self.octave}", (15, H-18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 200, 0), 2, cv2.LINE_AA)
        # Shortcuts
        cv2.putText(frame, "1-5:Scale  Up/Dn:Octave  B:Blur  R:Recal  Q:Quit", (300, H-18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (150, 150, 150), 1, cv2.LINE_AA)

        # Status line
        self.fps_n += 1
        el = time.time() - self.fps_t
        if el >= 1.0:
            self.fps_v = self.fps_n / el; self.fps_n = 0; self.fps_t = time.time()
        blur_s = "ON" if self.seg_enabled else "OFF"
        info = f"FPS: {self.fps_v:.0f}   Blur: {blur_s}   {self.native_w}x{self.native_h}"
        if self.depth_status and self.depth_status != "OK":
            ds_col = (0, 100, 255)  # orange warning
            cv2.putText(frame, f"DEPTH: {self.depth_status}", (W//2 - 120, H//2),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, ds_col, 2, cv2.LINE_AA)
        if self.active_note:
            info += f"   Note: {self.active_note}"
        cv2.putText(frame, info, (10, H-bot_h-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 150), 2, cv2.LINE_AA)

    def draw_calibration(self, frame, H, W):
        cx, cy = W//2, H//2
        bsz = 70
        cv2.rectangle(frame, (cx-bsz, cy-bsz), (cx+bsz, cy+bsz), (0, 255, 0), 3)
        cv2.line(frame, (cx-15, cy), (cx+15, cy), (0, 255, 0), 2)
        cv2.line(frame, (cx, cy-15), (cx, cy+15), (0, 255, 0), 2)
        msg = "Place palm in box, press C"
        (tw, _), _ = cv2.getTextSize(msg, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        cv2.putText(frame, msg, (cx - tw//2, cy - bsz - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
        oct_msg = f"Octave: {self.octave}  (Up/Down to change)"
        (tw2, _), _ = cv2.getTextSize(oct_msg, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 1)
        cv2.putText(frame, oct_msg, (cx - tw2//2, cy + bsz + 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 200, 0), 1, cv2.LINE_AA)
        return (cx-bsz, cy-bsz, bsz*2, bsz*2)

    # ---------- main loop ----------
    def run(self):
        win = "VisionHarp V4"
        cv2.namedWindow(win, cv2.WINDOW_NORMAL)
        cv2.resizeWindow(win, self.native_w, self.native_h)

        while True:
            ret, frame = self.cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            H, W = frame.shape[:2]

            if not self.is_calibrated:
                rect = self.draw_calibration(frame, H, W)
                key = cv2.waitKeyEx(1)
                if key == ord('c'): self.calibrate(frame, rect)
                if key == ord('q'): break
                if key == 2490368:  # Up arrow (Windows)
                    self.octave = min(self.octave + 1, self.max_octave); self._build_scale()
                if key == 2621440:  # Down arrow (Windows)
                    self.octave = max(self.octave - 1, self.min_octave); self._build_scale()
            else:
                # --- Detection on ORIGINAL frame (before blur) ---
                small = cv2.resize(frame, (self.PROC_W, self.PROC_H),
                                   interpolation=cv2.INTER_LINEAR)
                mask_s = self.build_mask_small(small)

                # --- Then apply blur segmentation to display frame ---
                if self.seg_enabled:
                    frame = self.segment_fg(frame, mask_s)

                # --- Contour detection ---
                cnts, _ = cv2.findContours(mask_s, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
                found = False; ai = -1

                if cnts:
                    c = max(cnts, key=cv2.contourArea)
                    c_area = cv2.contourArea(c)
                    if c_area > 800:
                        # --- Depth range check ---
                        area_lo = self.ref_area * self.depth_min
                        area_hi = self.ref_area * self.depth_max
                        if c_area < area_lo:
                            self.depth_status = "TOO FAR"
                        elif c_area > area_hi:
                            self.depth_status = "TOO CLOSE"
                        else:
                            self.depth_status = "OK"

                        if self.depth_status == "OK":
                            found = True
                            x, y, w, h = cv2.boundingRect(c)
                            roi = mask_s[y:y+h, x:x+w]
                            lp, pr = self.get_palm_center(roi)
                            gx = (x + lp[0]) * self.sx_ratio
                            gy = (y + lp[1]) * self.sy_ratio
                            pr_native = pr * self.sx_ratio

                            self.kf.predict()
                            self.kf.correct(np.array([[np.float32(gx)],[np.float32(gy)]]))
                            kx = float(self.kf.statePost[0])
                            ky = float(self.kf.statePost[1])

                            self.smooth_x += (kx - self.smooth_x) * self.ema_alpha
                            self.smooth_y += (ky - self.smooth_y) * self.ema_alpha
                            sx = int(np.clip(self.smooth_x, 0, W))
                            sy = int(np.clip(self.smooth_y, 0, H))

                            safe_r = max(1, min(200, int(pr_native)))

                            norm_x = sx / W
                            ai = self.get_note_index(norm_x, len(self.scale))
                            tf = self.scale[ai]
                            nn = self.note_names[ai]

                            nz = np.clip((safe_r - 10) / 50.0, 0, 1)
                            lfo = 2 ** (2.0 + nz * 6.0)
                            pan = np.clip(norm_x, 0, 1)
                            self.audio.update(tf, 0.5, lfo, 1.0, pan)
                            self.active_note = nn

                            color = (25, 255, 128)
                            self.draw_particles(frame, sx, sy, color)
                            cv2.circle(frame, (sx, sy), safe_r, color, 2)
                            cv2.putText(frame, nn, (sx+safe_r+10, sy+8),
                                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2, cv2.LINE_AA)

                if not found:
                    self.audio.update(440, 0, 4, 0.0, 0.5)
                    self.particles.clear()
                    self.active_note = ""

                self.draw_ui(frame, H, W, ai)

                key = cv2.waitKeyEx(1)
                if key == ord('q'): break
                if key == ord('r'):
                    self.is_calibrated = False
                    self.blurred_bg = None
                if key == ord('b'): self.seg_enabled = not self.seg_enabled
                if key == ord('1'): self.scale_mode="Major";      self._build_scale()
                if key == ord('2'): self.scale_mode="Minor";      self._build_scale()
                if key == ord('3'): self.scale_mode="Blues";      self._build_scale()
                if key == ord('4'): self.scale_mode="Pentatonic"; self._build_scale()
                if key == ord('5'): self.scale_mode="Hirajoshi";  self._build_scale()
                if key == 2490368:  # Up arrow (Windows)
                    self.octave = min(self.octave + 1, self.max_octave); self._build_scale()
                if key == 2621440:  # Down arrow (Windows)
                    self.octave = max(self.octave - 1, self.min_octave); self._build_scale()

            cv2.imshow(win, frame)

        self.audio.close(); self.cap.release(); cv2.destroyAllWindows()


if __name__ == "__main__":
    app = VisionHarpV4()
    app.run()


Calibrated — HSV mean: [103 143  82]  Octave: 3  Ref area: 14280  Res: 1920x1080
Calibrated — HSV mean: [103 165  86]  Octave: 3  Ref area: 19316  Res: 1920x1080
